# 从经典 Hopfield 到现代 Hopfield：MNIST 联想记忆实验

**[目标]**：用同一批 MNIST 图像，比较经典 Hopfield、二值 Dense Associative Memory、连续 Modern Hopfield 如何从下半部分遮挡的线索恢复原图。

**[来源]**：改写自 [Hopfield Networks is all you Need](https://ml-jku.github.io/hopfield-layers/) 的教学 notebook。

**[与 1982 二值实验的关系]**：经典 Hopfield 部分沿用同一条主线：储存输入 → Hebb 权重 → 残缺初态 → 异步更新 → 能量下降 → 吸引子。现代两种网络保留相同实验接口，但存储形式和检索规则不同。

**[阅读原则]**：图片只负责检查“检索出了什么”；曲线和轨迹负责回答“为什么得到这个结果”。


## 0. 实验管线：先看中文组件，再看代码入口

|阶段|中文组件|代码入口|与 1982 二值实验的对应|
|---|---|---|---|
|a. 储存输入|读取 MNIST、二值编码|`a1_load_mnist`、`a2_binarize_patterns`|对应“生成随机 ±1 记忆”|
|b. 权重矩阵 / 记忆表征|Hebb 权重、二值记忆表、连续记忆表|`b1_store_classical`、`b2_store_binary_dense`、`b3_store_continuous`|经典版本的 Hebb 权重相同；现代版本直接保存记忆|
|c. 检索输入初态|遮挡图像下半部分|`c1_make_binary_cue`、`c2_make_continuous_cue`|对应“从记忆出发并翻转若干 bit”|
|d. 动力学检索|异步能量下降、并行高阶读出、注意力读出|`d1_run_classical`、`d2_binary_dense_update`、`d3_continuous_update`|经典版本与 1982 一样逐神经元更新；现代版本通常一次读出|
|e. 测量指标|错误像素比例、MSE、能量、检索步数|`e1_binary_bit_error`、`e2_continuous_mse`、动力学记录|对应汉明距离、能量和收敛轮数|
|f. 展示|误差样本、检索结果、能量轨迹、记忆竞争|`f1`—`f5`|把 1982 的结果曲线扩展为图像检索图|

**[最短对照链]**：经典部分是 `a2 → b1 → c1 → d1 → e1 → f1/f2/f3`，与 1982 Notebook 的 `a1 → b1 → c3 → d1 → e1 → f1` 基本同构。


## 1. a：储存输入

MNIST 像素原本位于 `[0,1]`。经典网络与二值 Dense Associative Memory 使用 `{-1,+1}`；连续 Modern Hopfield 保留灰度值。

**[编码约定]**：有笔画的像素编码为 `-1`，白色背景编码为 `+1`。因此二值遮挡必须填成 `+1`，才表示“擦掉下半部分”，不能填成黑色笔画。

**[组件边界]**：这里仅把图像变成状态。符号判决属于后面的动力学，不混进输入编码。


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from torchvision import transforms

SEED = 0
torch.manual_seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")

# [环境] Colab 默认可能缺少中文字形；下载失败时仍可运行，只是标题可能缺字。
font_path = Path("NotoSansCJKtc-Regular.otf")
if not font_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(
            "https://github.com/notofonts/noto-cjk/raw/main/"
            "Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf",
            font_path,
        )
    except Exception:
        pass
if font_path.exists():
    fm.fontManager.addfont(font_path)
    plt.rcParams["font.family"] = "Noto Sans CJK TC"
plt.rcParams["axes.unicode_minus"] = False


def a1_load_mnist(
    batch_size: int = 1000,
    seed: int = SEED,
) -> tuple[torch.Tensor, torch.Tensor]:
    """[储存输入] 下载 MNIST；返回 images:(B,784)、labels:(B,)。"""
    dataset = torchvision.datasets.MNIST(
        root="./mnist_data",
        train=True,
        download=True,
        transform=transforms.ToTensor(),
    )
    generator = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
    )
    images, labels = next(iter(loader))
    return images.reshape(images.shape[0], 784).float(), labels


def a2_binarize_patterns(images: torch.Tensor) -> torch.Tensor:
    """[二值编码] [0,1] 灰度图 → {-1,+1} 图样。"""
    return torch.where(images > 0, -torch.ones_like(images), torch.ones_like(images))


images, labels = a1_load_mnist(batch_size=1000)
binary_images = a2_binarize_patterns(images)

print("连续图像矩阵：", tuple(images.shape))
print("二值图样矩阵：", tuple(binary_images.shape))
print("标签向量：", tuple(labels.shape))


## 2. b：权重矩阵 / 记忆表征

**[经典 Hopfield]**：`W = X.T @ X`。多条记忆叠加进同一个 `784×784` Hebb 权重矩阵；主对角线清零，去掉自连接。这与 1982 二值 Notebook 的 `b1` 相同。

**[二值 Dense Associative Memory]**：直接保存二值图样矩阵。检索时重新比较线索与每条记忆，没有单独的 `W`。

**[连续 Modern Hopfield]**：直接保存连续图样矩阵。检索时用 softmax 产生每条记忆的权重。

**[能量景观]**：经典网络的能量由 `W` 定义；“存入记忆”相当于用 Hebb 权重塑造低能吸引子。完整状态空间有 `2^784` 个点，无法直接绘制，后面只画检索实际经过的能量曲线和二维投影切片。


In [ ]:
def b1_store_classical(patterns: torch.Tensor) -> torch.Tensor:
    """[Hebb 存储] patterns:(n,D) → 对称权重 weights:(D,D)。"""
    weights = patterns.T @ patterns
    weights.fill_diagonal_(0.0)
    return weights


def b2_store_binary_dense(patterns: torch.Tensor) -> torch.Tensor:
    """[二值记忆表] 直接保留每条二值图样。"""
    return patterns.clone()


def b3_store_continuous(patterns: torch.Tensor) -> torch.Tensor:
    """[连续记忆表] 直接保留每条灰度图样。"""
    return patterns.clone()


## 3. c：检索输入初态

遮挡不是另一批数据，而是从目标记忆制造残缺初态。

- 二值模型：下半部分填成 `+1`，即当前编码中的白色背景。
- 连续模型：下半部分填成 `0`。

这对应 1982 Notebook 的 `c3_perturb_memory`：两者都从已存记忆制造不完整线索；区别只是那里随机翻转 bit，这里成块遮挡图像。


In [ ]:
def _reshape_image(pattern: torch.Tensor) -> torch.Tensor:
    """[形状约束] 本实验只处理 28×28 MNIST 图像。"""
    if pattern.numel() != 784:
        raise ValueError("pattern 必须包含 784 个像素")
    return pattern.reshape(28, 28)


def c1_make_binary_cue(pattern: torch.Tensor) -> torch.Tensor:
    """[二值检索初态] 擦掉图样的下半部分。"""
    cue = _reshape_image(pattern).clone()
    cue[14:, :] = 1
    return cue.flatten()


def c2_make_continuous_cue(pattern: torch.Tensor) -> torch.Tensor:
    """[连续检索初态] 擦掉图样的下半部分。"""
    cue = _reshape_image(pattern).clone()
    cue[14:, :] = 0
    return cue.flatten()


## 4. d：动力学检索

### d1 经典 Hopfield：逐神经元异步下降

每次只更新一个神经元，更新立即生效。对称 `W` 且无自连接时，每次真正翻转都会使

`E(state) = -0.5 × state.T @ W @ state`

单调不增。`d1_run_classical` 可选择保存每次翻转后的状态和能量，因此能看到检索用了多少轮、翻转多少步、沿什么路径到达固定点。这与 1982 Notebook 的 `d1_run_async` 相同。

### d2 二值 Dense Associative Memory：一次并行高阶读出

对每个输出 bit，比较它取 `+1` 与 `-1` 时对全部记忆的指数匹配分数。这里一次并行产生完整输出，没有经典网络那种逐神经元能量轨迹。

### d3 连续 Modern Hopfield：一次注意力读出

先计算 `attention = softmax(β · X @ cue)`，再用 `X.T @ attention` 读出。这里展示“哪些记忆参与竞争”，比套用经典二次能量更诚实。


In [ ]:
@dataclass
class ClassicalRun:
    """[动力学结果] 终态、扫描轮数、翻转计数与可选过程轨迹。"""

    final_state: torch.Tensor
    sweeps: int
    status: str
    flips_per_sweep: tuple[int, ...]
    state_history: list[torch.Tensor] | None = None
    energy_history: list[float] | None = None


def _classical_energy(weights: torch.Tensor, state: torch.Tensor) -> float:
    """[观测量] 经典对称 Hopfield 的二次能量。"""
    return float(-0.5 * state @ weights @ state)


def d1_run_classical(
    weights: torch.Tensor,
    cue: torch.Tensor,
    max_sweeps: int = 20,
    seed: int = SEED,
    record_states: bool = False,
    record_energy: bool = False,
) -> ClassicalRun:
    """[异步检索] 逐神经元更新，直到一整轮无翻转。"""
    state = cue.clone()
    generator = torch.Generator().manual_seed(seed)
    local_fields = weights @ state
    flips_per_sweep: list[int] = []
    state_history = [state.clone()] if record_states else None
    energy = _classical_energy(weights, state)
    energy_history = [energy] if record_energy else None

    for sweep in range(1, max_sweeps + 1):
        flips = 0
        order = torch.randperm(state.numel(), generator=generator)

        for i in order.tolist():
            field = float(local_fields[i])
            old_value = float(state[i])
            new_value = 1.0 if field > 0 else -1.0 if field < 0 else old_value

            if new_value != old_value:
                delta = new_value - old_value
                state[i] = new_value
                local_fields += weights[:, i] * delta
                flips += 1

                if record_states:
                    state_history.append(state.clone())
                if record_energy:
                    energy -= delta * field
                    energy_history.append(energy)

        flips_per_sweep.append(flips)
        if flips == 0:
            return ClassicalRun(
                state,
                sweep,
                "fixed",
                tuple(flips_per_sweep),
                state_history,
                energy_history,
            )

    return ClassicalRun(
        state,
        max_sweeps,
        "max_sweeps",
        tuple(flips_per_sweep),
        state_history,
        energy_history,
    )


def d2_binary_dense_update(
    memories: torch.Tensor,
    cue: torch.Tensor,
    temperature: float = 10.0,
) -> torch.Tensor:
    """[并行检索] 同时比较每个 bit 取 ±1 时的 Dense Memory 分数。"""
    z = cue.flatten()
    base_overlap = memories @ z

    positive_overlap = base_overlap[:, None] + memories * (1.0 - z)[None, :]
    negative_overlap = base_overlap[:, None] + memories * (-1.0 - z)[None, :]

    positive_score = torch.logsumexp(positive_overlap / temperature, dim=0)
    negative_score = torch.logsumexp(negative_overlap / temperature, dim=0)
    return torch.where(
        positive_score > negative_score,
        torch.ones_like(positive_score),
        -torch.ones_like(negative_score),
    )


def d3_continuous_update(
    memories: torch.Tensor,
    cue: torch.Tensor,
    beta: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """[注意力检索] softmax 加权读取连续记忆。"""
    similarities = memories @ cue
    attention = F.softmax(beta * similarities, dim=0)
    output = memories.T @ attention
    return output, attention


## 5. e / f：测量指标与读图工具

**[错误像素比例]**：二值输出与目标不同的像素数 ÷ 784。  
**[MSE]**：连续输出与目标的均方误差。  
**[能量]**：只对经典对称 Hopfield 使用 `-0.5 × state.T @ W @ state`。  
**[检索步数]**：经典网络同时报告扫描轮数和实际发生的神经元翻转数；Dense 与 Modern 当前各执行一次并行读出。  
**[二维景观]**：将高维状态轨迹和记忆投影到二维，再画该二维平面上的能量切片。它不是完整的 `2^784` 离散景观。


In [ ]:
def e1_binary_bit_error(
    output: torch.Tensor,
    target: torch.Tensor,
) -> float:
    """[测量] 二值输出的错误像素比例。"""
    return float(torch.mean((output != target).float()))


def e2_continuous_mse(
    output: torch.Tensor,
    target: torch.Tensor,
) -> float:
    """[测量] 连续输出的均方误差。"""
    return float(F.mse_loss(output, target))


def f1_plot_condition_samples(
    conditions,
    samples_by_condition,
    xlabel: str,
    ylabel: str,
    title: str,
) -> None:
    """[展示] 浅色点是单条检索样本，折线是各条件平均值。"""
    fig, ax = plt.subplots(figsize=(7, 4))
    positions = list(range(len(conditions)))
    means = []

    for position, samples in zip(positions, samples_by_condition):
        values = torch.tensor(samples, dtype=torch.float32)
        offsets = torch.linspace(-0.10, 0.10, max(len(samples), 2))[: len(samples)]
        ax.scatter(
            torch.full_like(values, float(position)) + offsets,
            values,
            alpha=0.45,
            color="tab:gray",
            label="单条目标记忆" if position == 0 else None,
        )
        means.append(float(values.mean()))

    ax.plot(
        positions,
        means,
        marker="o",
        linewidth=2,
        color="tab:blue",
        label="条件平均值",
    )
    ax.set_xticks(positions, [str(value) for value in conditions])
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.legend()
    plt.tight_layout()
    plt.show()


def f2_show_retrieval(
    target: torch.Tensor,
    cue: torch.Tensor,
    output: torch.Tensor,
    title: str,
) -> None:
    """[展示] 一个代表样本：目标、残缺初态、检索终态。"""
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    panels = [
        ("目标记忆", target),
        ("残缺初态", cue),
        ("检索终态", output),
    ]
    for ax, (panel_title, image) in zip(axes, panels):
        ax.imshow(image.reshape(28, 28), cmap="gray")
        ax.set_title(panel_title)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def f3_show_classical_energy_trajectory(
    run: ClassicalRun,
    weights: torch.Tensor,
    memories: torch.Tensor,
    target_index: int,
) -> None:
    """[展示] 精确能量序列 + 高维景观的二维投影切片。"""
    if run.state_history is None or run.energy_history is None:
        raise ValueError("需要 record_states=True 且 record_energy=True")

    states = torch.stack(run.state_history).float()
    memories = memories.float()
    cloud = torch.cat([states, memories], dim=0)
    center = cloud.mean(dim=0)
    _, _, vh = torch.linalg.svd(cloud - center, full_matrices=False)
    basis = vh[:2]

    state_xy = (states - center) @ basis.T
    memory_xy = (memories - center) @ basis.T
    all_xy = torch.cat([state_xy, memory_xy], dim=0)
    minima = all_xy.min(dim=0).values
    maxima = all_xy.max(dim=0).values
    padding = 0.12 * torch.clamp(maxima - minima, min=1.0)

    axis_1 = torch.linspace(float(minima[0] - padding[0]), float(maxima[0] + padding[0]), 80)
    axis_2 = torch.linspace(float(minima[1] - padding[1]), float(maxima[1] + padding[1]), 80)
    grid_1, grid_2 = torch.meshgrid(axis_1, axis_2, indexing="xy")
    coordinates = torch.stack([grid_1.flatten(), grid_2.flatten()], dim=1)

    quadratic = basis @ weights @ basis.T
    linear = basis @ (weights @ center)
    constant = center @ weights @ center
    landscape = -0.5 * (
        constant
        + 2 * (coordinates @ linear)
        + torch.einsum("bi,ij,bj->b", coordinates, quadratic, coordinates)
    )
    landscape = landscape.reshape(grid_1.shape)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(run.energy_history, color="tab:blue")
    axes[0].scatter(
        [0, len(run.energy_history) - 1],
        [run.energy_history[0], run.energy_history[-1]],
        color=["tab:orange", "tab:red"],
        zorder=3,
    )
    axes[0].set(
        xlabel="实际翻转步（只计状态发生变化）",
        ylabel="经典 Hopfield 能量 E",
        title="访问状态的精确能量轨迹",
    )

    contour = axes[1].contourf(
        grid_1.numpy(),
        grid_2.numpy(),
        landscape.numpy(),
        levels=28,
        cmap="viridis",
    )
    axes[1].plot(
        state_xy[:, 0],
        state_xy[:, 1],
        color="white",
        linewidth=1.5,
        alpha=0.9,
        label="检索轨迹",
    )
    axes[1].scatter(
        memory_xy[:, 0],
        memory_xy[:, 1],
        facecolors="none",
        edgecolors="white",
        s=45,
        label="已存记忆",
    )
    axes[1].scatter(
        state_xy[0, 0],
        state_xy[0, 1],
        marker="o",
        color="tab:orange",
        s=75,
        label="残缺初态",
    )
    axes[1].scatter(
        state_xy[-1, 0],
        state_xy[-1, 1],
        marker="X",
        color="tab:red",
        s=85,
        label="检索终态",
    )
    axes[1].scatter(
        memory_xy[target_index, 0],
        memory_xy[target_index, 1],
        marker="*",
        color="gold",
        edgecolors="black",
        s=180,
        label="目标记忆",
    )
    axes[1].set(
        xlabel="投影轴 1",
        ylabel="投影轴 2",
        title="能量景观二维切片与投影轨迹",
    )
    axes[1].legend(fontsize=8, loc="best")
    fig.colorbar(contour, ax=axes[1], label="二维切片能量")
    fig.suptitle(
        f"经典检索：{run.sweeps} 轮扫描，{sum(run.flips_per_sweep)} 次实际翻转"
    )
    plt.tight_layout()
    plt.show()


def f4_show_memory_competition(
    scores: torch.Tensor,
    target_index: int,
    ylabel: str,
    title: str,
    top_k: int = 10,
) -> None:
    """[展示] 检索时竞争最强的若干条记忆。"""
    top_k = min(top_k, scores.numel())
    values, indices = torch.topk(scores, k=top_k)
    colors = ["tab:orange" if int(index) == target_index else "tab:blue" for index in indices]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([str(int(index)) for index in indices], values, color=colors)
    ax.set(
        xlabel="记忆编号",
        ylabel=ylabel,
        title=title,
    )
    ax.text(
        0.99,
        0.96,
        "橙色 = 目标记忆",
        transform=ax.transAxes,
        ha="right",
        va="top",
    )
    plt.tight_layout()
    plt.show()


def f5_show_beta_effect(
    beta_values,
    errors,
    max_attention,
) -> None:
    """[展示] β 同时怎样改变检索误差与注意力集中度。"""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(beta_values, errors, marker="o")
    axes[0].set(xlabel="β", ylabel="MSE", title="同一目标的检索误差")
    axes[1].plot(beta_values, max_attention, marker="o", color="tab:orange")
    axes[1].set(
        xlabel="β",
        ylabel="最大注意力权重",
        title="记忆选择的集中程度",
    )
    plt.tight_layout()
    plt.show()


## 6. 公共组件自检

先验证组件边界，再运行 MNIST 实验。这里重点检查：Hebb 权重对称、无自连接；经典异步更新的能量单调不增；现代读出的形状和注意力归一化正确。


In [ ]:
toy_memories = torch.tensor(
    [[1.0, 1.0, -1.0, -1.0], [-1.0, 1.0, -1.0, 1.0]]
)
toy_weights = b1_store_classical(toy_memories)

assert torch.allclose(toy_weights, toy_weights.T)
assert torch.allclose(torch.diag(toy_weights), torch.zeros(4))

toy_cue = toy_memories[0].clone()
toy_cue[0] *= -1
toy_run = d1_run_classical(
    toy_weights,
    toy_cue,
    max_sweeps=10,
    record_states=True,
    record_energy=True,
)
assert toy_run.energy_history is not None
assert all(
    later <= earlier + 1e-6
    for earlier, later in zip(toy_run.energy_history, toy_run.energy_history[1:])
)

toy_dense = d2_binary_dense_update(toy_memories, toy_memories[0])
assert set(torch.unique(toy_dense).tolist()) <= {-1.0, 1.0}

toy_output, toy_attention = d3_continuous_update(
    toy_memories,
    toy_memories[0],
    beta=1.0,
)
assert toy_output.shape == toy_memories[0].shape
assert torch.allclose(toy_attention.sum(), torch.tensor(1.0))

print("公共组件自检通过：经典能量单调、Dense 输出二值、Modern 注意力和为 1")


## 7. 实验 1：经典 Hopfield——误差、步数与能量轨迹

**[问题]**：存储 MNIST 图样数从 `1 → 3 → 10` 时，检索误差怎样变化？一次具体检索经过多少次神经元翻转？能量如何下降到吸引子？

**[配方]**：`二值编码 → Hebb 存储 → 遮挡初态 → 异步更新 → 错误像素比例 / 能量 → 三类图`

**[自变量]**：存储图样数 `n`。  
**[样本单位]**：一条目标记忆的一次检索。每个条件评估 `min(n,10)` 条目标记忆。  
**[图中数据]**：浅色点是一条目标的错误像素比例；蓝线是该条件的平均值。`n=10` 的第 0 条目标额外保存逐次翻转轨迹。


In [ ]:
classical_n_values = [1, 3, 10]
classical_error_samples = []
classical_example = None
classical_trace_bundle = None

for n in classical_n_values:
    memories = binary_images[:n]
    weights = b1_store_classical(memories)
    errors = []

    for target_index in range(min(n, 10)):
        target = memories[target_index]
        cue = c1_make_binary_cue(target)
        record_trace = n == 10 and target_index == 0
        run = d1_run_classical(
            weights,
            cue,
            max_sweeps=20,
            seed=1000 + 100 * n + target_index,
            record_states=record_trace,
            record_energy=record_trace,
        )
        errors.append(e1_binary_bit_error(run.final_state, target))

        if record_trace:
            classical_example = (target, cue, run.final_state)
            classical_trace_bundle = (
                run,
                weights.clone(),
                memories.clone(),
                target_index,
            )

    classical_error_samples.append(errors)
    print(
        f"经典 Hopfield：n={n}，样本数={len(errors)}，"
        f"平均错误率={sum(errors) / len(errors):.3f}"
    )

f1_plot_condition_samples(
    classical_n_values,
    classical_error_samples,
    xlabel="存储图样数 n",
    ylabel="错误像素比例",
    title="经典 Hopfield：每条目标的误差与条件平均值",
)
f2_show_retrieval(
    *classical_example,
    title="经典 Hopfield 代表样本（n=10，第 0 条目标）",
)
f3_show_classical_energy_trajectory(*classical_trace_bundle)

trace_run = classical_trace_bundle[0]
print(
    "代表轨迹："
    f"状态={trace_run.status}，扫描={trace_run.sweeps} 轮，"
    f"每轮翻转数={trace_run.flips_per_sweep}，"
    f"合计实际翻转={sum(trace_run.flips_per_sweep)} 次"
)


**[图 1：误差分布]**：横轴 `n` 是写入同一权重矩阵的记忆数；每个浅色点对应一条目标记忆。蓝线不是理论容量，只是当前固定 MNIST 样本和随机更新日程下的条件平均值。

**[图 2：三联图]**：只回答代表样本最终恢复成什么样，不用于证明总体性能。

**[图 3 左：精确能量轨迹]**：纵轴是每个实际访问的二值状态的真实能量；横轴只计发生状态改变的神经元翻转。对称权重和异步更新保证它不升高。

**[图 3 右：二维能量切片]**：圆圈是已存记忆，星号是目标，白线是高维状态轨迹投影。背景仅是轨迹所在二维平面的连续切片，不是完整 `2^784` 状态空间，也不能据此读出真实吸引盆体积。


## 8. 实验 2：二值 Dense Associative Memory——一次读出与记忆竞争

**[问题]**：改用高阶指数匹配后，`n=10` 与 `n=100` 的检索误差如何变化？残缺线索最接近哪些已存记忆？

**[配方]**：`二值编码 → 二值记忆表 → 遮挡初态 → 一次并行读出 → 错误像素比例 / 初始重叠度`

**[样本单位]**：每个条件的前 10 条目标记忆。误差图中的每个浅色点是一条目标；记忆竞争图只分析 `n=100` 的第 0 条目标。


In [ ]:
dense_n_values = [10, 100]
dense_error_samples = []
dense_example = None
dense_competition = None

for n in dense_n_values:
    memories = b2_store_binary_dense(binary_images[:n])
    errors = []

    for target_index in range(10):
        target = memories[target_index]
        cue = c1_make_binary_cue(target)
        output = d2_binary_dense_update(memories, cue)
        errors.append(e1_binary_bit_error(output, target))

        if n == 100 and target_index == 0:
            dense_example = (target, cue, output)
            normalized_overlap = (memories @ cue) / cue.numel()
            dense_competition = (normalized_overlap, target_index)

    dense_error_samples.append(errors)
    print(
        f"二值 Dense Memory：n={n}，样本数={len(errors)}，"
        f"平均错误率={sum(errors) / len(errors):.3f}，检索步数=1 次并行读出"
    )

f1_plot_condition_samples(
    dense_n_values,
    dense_error_samples,
    xlabel="存储图样数 n",
    ylabel="错误像素比例",
    title="二值 Dense Memory：每条目标的误差与条件平均值",
)
f2_show_retrieval(
    *dense_example,
    title="二值 Dense Memory 代表样本（n=100，第 0 条目标）",
)
f4_show_memory_competition(
    *dense_competition,
    ylabel="与残缺初态的归一化重叠度",
    title="并行读出前：竞争最强的 10 条二值记忆",
)


**[误差图]**：比较的是两种存储规模下，10 条目标各自的最终错误像素比例。均值上升表示更多相关 MNIST 图样参与竞争后，检索更容易混淆。

**[记忆竞争图]**：柱高是残缺初态与每条记忆的归一化重叠度；它解释“哪些记忆一开始最像线索”。真正输出仍由 `d2_binary_dense_update` 对每个 bit 比较高阶指数分数，不应把柱高直接当作最终概率。

**[与 1982 的区别]**：这里不是沿二次能量逐 bit 下坡，而是一次并行高阶读出。因此没有经典网络那种多步能量轨迹；强行画 `-0.5 state.T @ W @ state` 反而会使用一个并不存在的 `W`。


## 9. 实验 3：连续 Modern Hopfield——误差与注意力竞争

**[问题]**：连续检索在 `n=10、100、1000` 时的 MSE 如何变化？一次读出把权重集中到哪些记忆？

**[配方]**：`连续图像 → 连续记忆表 → 遮挡初态 → 一次注意力读出 → MSE / 注意力权重`

**[样本单位]**：每个条件评估前 `min(n,20)` 条目标。误差图中的浅色点是一条目标；注意力图只分析 `n=1000` 的第 0 条目标，默认 `β=8`。


In [ ]:
continuous_n_values = [10, 100, 1000]
continuous_error_samples = []
continuous_example = None
continuous_competition = None
beta = 8.0

for n in continuous_n_values:
    memories = b3_store_continuous(images[:n])
    errors = []

    for target_index in range(min(n, 20)):
        target = memories[target_index]
        cue = c2_make_continuous_cue(target)
        output, attention = d3_continuous_update(memories, cue, beta=beta)
        errors.append(e2_continuous_mse(output, target))

        if n == 1000 and target_index == 0:
            continuous_example = (target, cue, output)
            continuous_competition = (attention, target_index)

    continuous_error_samples.append(errors)
    print(
        f"连续 Modern Hopfield：n={n}，样本数={len(errors)}，"
        f"平均 MSE={sum(errors) / len(errors):.6f}，检索步数=1 次注意力读出"
    )

f1_plot_condition_samples(
    continuous_n_values,
    continuous_error_samples,
    xlabel="存储图样数 n",
    ylabel="MSE",
    title="连续 Modern Hopfield：每条目标的 MSE 与条件平均值",
)
f2_show_retrieval(
    *continuous_example,
    title="连续 Modern Hopfield 代表样本（n=1000，第 0 条目标）",
)
f4_show_memory_competition(
    *continuous_competition,
    ylabel="注意力权重",
    title="一次读出中权重最高的 10 条连续记忆",
)


**[误差图]**：每个浅色点是一条目标的一次检索 MSE；蓝线是同一存储规模下最多 20 条目标的平均值。它测的是当前相关 MNIST 样本，不是随机模式理论容量。

**[注意力图]**：柱高是一次读出中每条记忆获得的 softmax 权重，权重总和为 1。橙色目标柱接近 1 表示线索几乎唯一选中目标；若多个柱同时较高，输出就是多条记忆的混合。

**[与 1982 的区别]**：1982 网络把记忆写进突触权重并逐神经元演化；这里把记忆保留为表，先算相似度，再一次加权读出。两者都有“从线索回忆记忆”的功能，但中间机制不同。


## 10. 实验 4：β 怎样改变一次注意力读出？

**[问题]**：固定 100 条记忆和同一条残缺初态时，β 如何同时改变检索误差与注意力集中程度？

**[自变量]**：`β = 0.1、0.2、0.5、1、2、4、8`。  
**[因变量 1]**：输出与目标的 MSE。  
**[因变量 2]**：100 条记忆中最大的注意力权重。  
**[每个点的含义]**：同一个目标、同一个残缺初态，在一个 β 条件下的一次读出；这里没有重复试验或误差条。


In [ ]:
beta_values = [0.1, 0.2, 0.5, 1, 2, 4, 8]
beta_memories = b3_store_continuous(images[:100])
beta_target = beta_memories[0]
beta_cue = c2_make_continuous_cue(beta_target)
beta_errors = []
beta_max_attention = []

for beta in beta_values:
    output, attention = d3_continuous_update(
        beta_memories,
        beta_cue,
        beta=beta,
    )
    error = e2_continuous_mse(output, beta_target)
    maximum = float(attention.max())
    beta_errors.append(error)
    beta_max_attention.append(maximum)
    print(f"β={beta:>3}，MSE={error:.6f}，最大注意力权重={maximum:.4f}")

f5_show_beta_effect(
    beta_values,
    beta_errors,
    beta_max_attention,
)


## 11. 与 1982 二值 Notebook 对照

**[相同主线]**：两边都有“储存输入 → 构造检索初态 → 执行动力学 → 测量终态”的实验骨架。

**[经典部分几乎相同]**：二值状态、Hebb 权重、对角线清零、异步逐神经元更新、二次能量单调下降、最终落入固定点。这里的遮挡图像相当于 1982 实验中的受扰动记忆。

**[现代部分不同]**：Dense 与 Modern Hopfield 直接保存记忆表。前者一次并行比较高阶匹配分数；后者一次计算注意力并加权读出。它们不应套用经典网络的 `W` 和二次能量图。

**[四类图分别回答什么]**：

1. 误差散点和均值线：该条件下多条目标总体检索得怎样。
2. 三联图：某一条代表样本恢复成什么样。
3. 经典能量轨迹：状态用了多少次实际翻转，能量是否持续下降。
4. 记忆竞争柱状图：Dense/Modern 在一次读出时，哪些记忆最接近或获得最大权重。

因此，真正与 1982 能量景观直接对照的是实验 1；实验 2、3 展示的是现代网络替代逐步能量下降的“直接记忆竞争”机制。
